# Dataset Analysis — Benign Sample Construction

This notebook builds the balanced evaluation dataset by sampling benign snippets proportionally from all 20 benchmark projects.
It is a preprocessing step used before running the main evaluation in `VulnFixAI.ipynb`.

In [ ]:
import os
import pandas as pd
from pathlib import Path

# All paths are relative to this notebook's location
NOTEBOOK_DIR = Path(os.path.abspath(""))

# Primary dataset — update filename if yours differs
DATASET_PATH = NOTEBOOK_DIR / "Trining DataSet" / "all_vulnerabilities_Training_Dataset.csv"

# Fallback: look for the file anywhere in the training dataset folder
if not DATASET_PATH.exists():
    candidates = list((NOTEBOOK_DIR / "Trining DataSet").glob("*.csv"))
    if candidates:
        DATASET_PATH = candidates[0]
        print(f"[Info] Using fallback dataset: {DATASET_PATH.name}")
    else:
        raise FileNotFoundError(
            "No CSV found in 'Trining DataSet/'. "
            "Place your training CSV there and re-run."
        )

print(f"Loading dataset from: {DATASET_PATH}")
df = pd.read_csv(DATASET_PATH)
print(f"Shape: {df.shape}")
print(f"Columns: {list(df.columns)}")

In [ ]:
# Filter benign rows
benign_df = df[df['Status'] == 'benign']
print(f"Total benign rows: {len(benign_df)}")
print(f"\nBenign rows per project:")
print(benign_df['Project Name'].value_counts())

In [ ]:
# Sample 10,000 benign rows proportionally from all projects
# This ensures every project is represented in proportion to its size.

TARGET = 10_000

if len(benign_df) >= TARGET:
    project_counts = benign_df['Project Name'].value_counts()
    sampled_rows = []

    for project in project_counts.index:
        project_data = benign_df[benign_df['Project Name'] == project]
        proportion   = len(project_data) / len(benign_df)
        sample_size  = int(proportion * TARGET)
        if sample_size > 0:
            n = min(len(project_data), sample_size)
            sampled_rows.append(project_data.sample(n=n, random_state=42))

    benign_10k = pd.concat(sampled_rows, ignore_index=True)

    # Fill up to exactly TARGET if proportional rounding left a shortfall
    if len(benign_10k) < TARGET:
        remaining   = TARGET - len(benign_10k)
        extra       = benign_df[~benign_df.index.isin(benign_10k.index)]
        additional  = extra.sample(n=min(remaining, len(extra)), random_state=42)
        benign_10k  = pd.concat([benign_10k, additional], ignore_index=True)
    elif len(benign_10k) > TARGET:
        benign_10k = benign_10k.sample(n=TARGET, random_state=42)

    print(f"Sampled {len(benign_10k)} benign rows")
    print(f"\nDistribution across projects:")
    print(benign_10k['Project Name'].value_counts())
else:
    print(f"Only {len(benign_df)} benign rows available (< {TARGET}) — using all.")
    benign_10k = benign_df

In [ ]:
# Save the sampled benign rows — output goes next to this notebook
OUT_PATH = NOTEBOOK_DIR / "benign_10k_sampled.csv"
benign_10k.to_csv(OUT_PATH, index=False)
print(f"Saved {len(benign_10k)} rows to: {OUT_PATH}")